# Module 9 — Audit metrics, code, and claims

Predict first, then run and inspect. Record your reasoning in the response cells. The worked notebook is a separate review resource. The class notebook is not an additional homework submission.

Run `uv sync --frozen` in this repo and select its `.venv` kernel.


In [ ]:
import pandas as pd
import random
from itertools import permutations

print("Ready for the four examples.")

## 1 · Customer loyalty
**Request:** “Show which store has the most loyal customers.”

Before coding, ask: Who uses the result? What decision follows? Does loyalty mean frequent orders, repeat visits, spending, or something else?

For this example, all rows are distinct completed orders in January 2026. Customer IDs are reliable; every order belongs to one store. We compare repeat purchasing during this month, not long-run retention or a causal effect of the store.

In [ ]:
orders = pd.DataFrame({
    "order": ["o1", "o2", "o3", "o4", "o5", "o6"],
    "store": ["A", "A", "A", "A", "B", "B"],
    "customer": ["Ana", "Ana", "Ana", "Ben", "Cai", "Cai"],
    "date": ["2026-01-02", "2026-01-03", "2026-01-04",
             "2026-01-02", "2026-01-02", "2026-01-05"]
})
print(orders.to_string(index=False))

### Pause and predict
1. Calculate each store's average orders per customer.
2. Calculate the share of its customers with at least two orders.
3. Do these measures rank the stores in the same way?
4. Could a newly acquired customer be less likely to qualify simply because they joined late?

Use `talk-it-through`, if available, to discuss the choices one question at a time. Without the skill, ask an agent: “Help me define this metric before implementing it. Ask one question at a time about the decision, population, denominator, time window, and assumptions. Ask me to predict a tiny example before recommending a definition. Summarize our agreement before coding.”

**Your prediction and reasoning:**



In [ ]:
per_customer = (orders.groupby(["store", "customer"])
                .size().reset_index(name="n_orders"))
print(per_customer.to_string(index=False))

### Pause and trace the incorrect code
The following code runs. What does its denominator count? Predict both outputs.

**Your prediction and reasoning:**



In [ ]:
repeat = per_customer[per_customer["n_orders"] >= 2]
wrong_share = repeat.groupby("store").size() / orders.groupby("store").size()
print(wrong_share)

### Your check
Explain the denominator error. Write a corrected function. Then add a store with two one-order customers and check whether it remains in the report. Explain how a store with no orders differs.

**Your reasoning:**


In [ ]:
def repeat_customer_share(order_rows):
    # Your correction
    pass

## 2 · Average tip per visit
**Requirement:** report average tip per visit. A blank tip explicitly means the customer left no tip. Each visit has at least one service.

**Pause and predict:** Visit A has a \$10 tip and two services; visit B has no tip and one service. What is the mean tip per visit? What happens if we calculate the mean after the merge?

**Your prediction and reasoning:**



In [ ]:
visits = pd.DataFrame({"visit": ["A", "B"], "tip": [10.0, None]})
services = pd.DataFrame({"visit": ["A", "A", "B"],
                         "service": ["haircut", "coloring", "haircut"]})
print("Visits:\n", visits.to_string(index=False))
print("\nServices:\n", services.to_string(index=False))

In [ ]:
clean_visits = visits.copy()
clean_visits["tip"] = clean_visits["tip"].fillna(0)
merged = services.merge(clean_visits, on="visit", validate="many_to_one")
print(merged.to_string(index=False))
wrong_mean_tip = merged["tip"].mean()
print("Mean after merge:", round(wrong_mean_tip, 2))

### Your check
Explain the unit represented by each merged row. Compute the visit-level mean. Add another service to visit A and compare the two calculations.

**Your reasoning:**


In [ ]:
## Write and run your correction here.

### Change the assumption
What if a blank tip means not recorded rather than no tip? State what you can calculate and what remains unknown.

**Your reasoning:**


## 3 · Delivery performance
**Request:** “Show our overall on-time delivery rate.”

For the numeric example, each total counts completed deliveries with known outcomes, and each delivery belongs to exactly one store. The on-time rule is already fixed. Later, we will question those assumptions.

In [ ]:
delivery = pd.DataFrame({"store": ["A", "B"],
                         "on_time": [9, 1], "total": [10, 2]})
print(delivery.to_string(index=False))

### Pause and predict
1. What is each store's on-time rate?
2. What is the average of those two rates?
3. What share of all deliveries were on time?
4. Which result gives each store equal weight? Which gives each delivery equal weight?

**Your prediction and reasoning:**



In [ ]:
delivery["rate"] = delivery["on_time"] / delivery["total"]
average_store_rate = delivery["rate"].mean()
overall_delivery_rate = delivery["on_time"].sum() / delivery["total"].sum()
print(delivery.to_string(index=False))
print(f"Average store rate: {average_store_rate:.2%}")
print(f"Overall delivery rate: {overall_delivery_rate:.2%}")

### Your check
Which result matches the stated overall rate? Verify it with twelve individual delivery outcomes. Explain why your second route is independent.

**Your reasoning:**


In [ ]:
## Write your independent check here.

### New check before running
Suppose A's records are split into two batches, each with 5 deliveries: one with 5 on-time deliveries and one with 4. B is unchanged. Does the overall delivery rate change? Should it?

### Your check
Calculate both rates after the split and explain which result should stay unchanged.

**Your reasoning:**


In [ ]:
batches = pd.DataFrame({"batch": ["A1", "A2", "B"],
                        "on_time": [5, 4, 1], "total": [5, 5, 2]})
# Your comparison

## 4 · A simplified weighted lottery
This is a hypothetical lottery, not a model of current immigration policy.

Person A holds 1 entry and person B holds 4 entries. Shuffle all entries uniformly. Scan the line, keeping only each person's first entry, until the available slots are filled. A person can win at most once. Slots never exceed the number of people.

An app uses `slots × person's entries / total entries` as the person's probability of selection.

### Pause and predict
With one slot, what are the two probabilities? With two slots? Can a probability exceed 1?

**Your prediction and reasoning:**



In [ ]:
def linear_odds(weights, slots):
    return [slots * weight / sum(weights) for weight in weights]

print("One slot:", linear_odds([1, 4], 1))
print("Two slots:", linear_odds([1, 4], 2))

### Your check
Predict one-slot and two-slot results by hand. Explain what the formula assumes. Describe a simulation before reading the supplied reference implementation.

**Your reasoning:**


In [ ]:
## Optional: write your simulation or hand checks here.

### Checks for the supplied simulation
Use the worked notebook’s simulation after discussing your prediction. Check zero slots, enough slots for everyone, and equal weights with one slot. Explain what these checks establish and what they cannot establish about a real lottery.

**Your reasoning:**


### Optional extension
The worked notebook enumerates all permutations of five labeled tickets to check the simulation exactly. Read this only after predicting the tiny case; enumeration is practical here because the example is small.


## Apply the audit
Choose one number in your team's app:
1. What exactly should it measure?
2. What tiny case would help you check it?
3. Which assumption or line deserves inspection?
4. What evidence would justify a change?
